In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

In [5]:
"""
扩散指数计算模块

此模块提供了一个核心函数用于计算扩散指数。
扩散指数用于衡量经济指标的变化趋势，取值范围为0-100。

主要功能：
- 计算扩散指数（支持同比、环比和组合计算）
- 支持自定义变化判定阈值
- 返回标准化的扩散指数值（0-100）
"""

def preprocess_data(file_path: str) -> pd.DataFrame:
    """
    预处理黑色金属冶炼行业数据
    
    参数:
    file_path: str
        Excel文件路径，包含weekly sheet
    
    返回:
    pd.DataFrame: 处理后的数据框，包含时间索引
    """
    try:
        # 读取Excel文件中的weekly sheet
        df = pd.read_excel(file_path, sheet_name='weekly')
        
        # 确保第一列是日期列
        if not pd.api.types.is_datetime64_any_dtype(df.iloc[:, 0]):
            df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
        
        # 将第一列设置为索引
        df.set_index(df.columns[0], inplace=True)

        # 确保数据按时间排序
        df.sort_index(inplace=True)
        
        return df
        
    except Exception as e:
        raise ValueError(f"数据预处理失败: {str(e)}")

def calculate_mixDI(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("输入数据框必须包含时间索引")
    
    # 确保数据按时间排序
    df = df.sort_index()

    # +++ 新增：将所有 0 值替换为 NaN，视为缺失 +++
    df = df.replace(0, np.nan)

    # 创建结果DataFrame，使用时间索引
    result_df = pd.DataFrame(index=df.index)
    
    # 计算组合变化率：先计算同比变化，再计算环比变化
    # 1. 计算同比变化
    yoy_relative = df.pct_change(52, fill_method=None)
    
    # 2. 计算同比变化的环比变化
    combined_changes = yoy_relative.pct_change(periods=4, fill_method=None)
    
    # 分别检查每种变化率的缺失值比例
    def check_missing_ratio(changes):
        missing_ratio = changes.isna().mean(axis=1)
        return missing_ratio <= 0.33  # 缺失值不超过三分之一
    
    valid_weeks = check_missing_ratio(combined_changes)
    
    # 按周计算扩散指数
    def calculate_weekly_di(changes, valid_weeks):
        weekly_di = []
        
        for i, week in enumerate(changes.index):
            if not valid_weeks.iloc[i]:
                weekly_di.append(np.nan)
                continue
                
            week_data = changes.loc[week]
            improvements = (week_data > 0).sum()
            total = week_data.count()  # 非NaN的数量
            
            if total == 0: # 避免除以零
                di_value = np.nan
            else:
                di_value = (improvements / total) * 100
            
            weekly_di.append(round(di_value, 2) if not np.isnan(di_value) else np.nan) # 处理 NaN 情况
                
        return weekly_di
    
    result_df['mix_DI'] = calculate_weekly_di(combined_changes, valid_weeks)
    
    return result_df

In [6]:
def plot_mixDI(
    di_df: pd.DataFrame,
    save_dir: str = None
) -> None:
    """
    绘制交互式 Mix_DI (同环比) 扩散指数时间序列图。

    参数:
    di_df: pd.DataFrame
        包含扩散指数的数据框，必须包含'mix_DI'列
    save_dir: str, 默认值 None
        图表保存目录，如果为None则不保存
    """
    # --- 直接定义要绘制的指数信息 ---
    index_type = 'mix_DI'
    title = '同环比扩散指数'
    label = '同环比'
    color = '#2ca02c'  # 绿色
    # --- 结束定义 ---

    # 创建保存目录
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # --- 移除 for 循环 ---
    # 创建图形
    fig = go.Figure()

    # 检查列是否存在
    if index_type not in di_df.columns:
        print(f"错误: DataFrame 中未找到列 '{index_type}'")
        return

    # 添加扩散指数线
    fig.add_trace(go.Scatter(
        x=di_df.index,
        y=di_df[index_type],
        name=label,
        mode='lines+markers',
        line=dict(width=2, color=color),
        marker=dict(size=4, color=color)
    ))

    # 添加中性线
    fig.add_hline(y=50, line_dash="dash", line_color="red", opacity=0.5)

    # 更新布局
    fig.update_layout(
        title=title, # 使用定义的标题
        yaxis_title="扩散指数",
        yaxis=dict(range=[0, 100]),
        showlegend=True,
        hovermode='x unified',
        template='plotly_white'
    )

    # 更新x轴
    fig.update_xaxes(
        rangeslider_visible=False, 
        rangeselector=dict(  # 添加范围选择器
            buttons=list([
                dict(count=6, label="6月", step="month", stepmode="backward"),
                dict(count=1, label="1年", step="year", stepmode="backward"),
                dict(count=3, label="3年", step="year", stepmode="backward"),
                dict(step="all", label="全部")
            ])
        )
    )

    # 保存图表
    if save_dir:
        save_path = f"{save_dir}/{title}.html"
        fig.write_html(save_path)
        print(f"图表已保存至: {save_path}")

    # 显示图表
    fig.show()


In [11]:

os.getcwd()
file = os.getcwd() + "\\化学原料和化学制品制造业.xlsx"
df = preprocess_data(file)
mixDi = calculate_mixDI(df)
mixDi


C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


,mix_DI
date,
2020-01-02,NaN
2020-01-09,NaN
2020-01-16,NaN
2020-01-23,NaN
2020-01-30,NaN
...,...
2025-02-20,23.33
2025-02-27,NaN
2025-03-06,30.00
